# 05 - LLM-as-Judge Results

Reproduces the paper's LLM-as-judge tables:

- **Table 3** - Haiku 4-condition Pearson r on SDA, ST_sent, ST_para, D-Wiki
- **Table 4** - 5-model direct-scoring Pearson r, output_only, on the same 4 datasets
- **Table 5** - ARTS + LR-ARTS Pearson r (Haiku + other 4 models)
- **Table 10** - Full direct-scoring grid (5 models x 7 datasets x conditions)
- **Table 11** - LR-ARTS Spearman + Kendall (absolute)
- **Table 12** - Pairwise prompting: r, delta-r vs direct, position bias
- **Tables 13 / 14** - 95% confidence intervals for ARTS and human-annotated datasets

Each table is also persisted to `results/tables/` as CSV.

**Conventions**

- Tables 3, 4, 5, 10, 11, 13, 14 use **absolute** Pearson / Spearman / Kendall correlations (matches the paper PDF).
- Table 12 reports **signed** Pearson r (so delta-r is meaningful) plus the manifest-derived position-bias rate.
- Ministral-8B pairwise on SDA shows '-' because Bradley-Terry could not fit (the model returned empty on ~85% of the paper's pairwise prompts). The position-bias column still reports the bias rate.

In [ ]:
import json
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd

REPO_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(REPO_ROOT))
os.chdir(REPO_ROOT)

from src.data_loader import (
    load_arts_datasets,
    load_lr_arts_datasets,
    load_human_annotated_datasets,
    load_sda,
)
from src.statistics.correlation import compute_correlation

OUTPUT_DIR = Path('results/tables')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Jupyter-aware display: rich tables in the notebook, fallback to print() in a script.
try:
    from IPython.display import display
except Exception:
    def display(x):
        print(x)

print(f'Working from: {REPO_ROOT}')
print(f'Tables will be written to: {OUTPUT_DIR}')

In [ ]:
# Load all datasets once.
arts = load_arts_datasets()
lr_arts = load_lr_arts_datasets()
human = {'SDA': load_sda(), **load_human_annotated_datasets()}  # SDA + ST-sent + ST-para + D-Wiki

# Convenience: column-name display order matching the paper.
HAIKU_CONDITIONS = [
    'Claude(simp)',
    'Claude(simp+src)',
    'Claude(simp+ref)',
    'Claude(simp+src+ref)',
]
MODELS = [
    'Claude(simp)',
    'GPT-4o-mini',
    'Gemma-4-E4B',
    'Ministral-8B',
    'Llama-3.1-8B',
]
PAIRWISE_MODELS = [
    'GPT-4o-mini (pairwise)',
    'Gemma-4-E4B (pairwise)',
    'Ministral-8B (pairwise)',
    'Llama-3.1-8B (pairwise)',
]
HUMAN_DATASETS = ['SDA', 'ST-sent', 'ST-para', 'D-Wiki']

# Pairwise position-bias values (refreshed by scripts/build_release_data.py).
with open('data/pairwise_position_bias.json') as f:
    position_bias = json.load(f)


def corr(data, metric, method='pearson', use_abs=True):
    """Compute |r| (default) or signed r between metric and human.
    Returns NaN if the metric is missing or all-NaN."""
    if metric not in data['metric_scores']:
        return float('nan')
    x = np.asarray(data['metric_scores'][metric], dtype=float)
    y = np.asarray(data['human_scores'], dtype=float)
    mask = ~(np.isnan(x) | np.isnan(y))
    if mask.sum() < 3:
        return float('nan')
    res = compute_correlation(x, y, method=method)
    return res['abs_correlation'] if use_abs else res['correlation']


def corr_with_ci(data, metric, method='pearson', use_abs=True, decimals=3):
    """Return a formatted 'r [lo, hi]' string for the metric, or '-' if missing."""
    if metric not in data['metric_scores']:
        return '-'
    x = np.asarray(data['metric_scores'][metric], dtype=float)
    y = np.asarray(data['human_scores'], dtype=float)
    mask = ~(np.isnan(x) | np.isnan(y))
    if mask.sum() < 3:
        return '-'
    res = compute_correlation(x, y, method=method)
    r = abs(res['correlation']) if use_abs else res['correlation']
    lo = abs(res['ci_lower']) if use_abs else res['ci_lower']
    hi = abs(res['ci_upper']) if use_abs else res['ci_upper']
    if use_abs and lo > hi:
        lo, hi = hi, lo  # bounds may flip after taking absolute value
    return f'{r:.{decimals}f} [{lo:.{decimals}f}, {hi:.{decimals}f}]'

## Table 3 - Haiku 4 conditions on the human-annotated datasets

Absolute Pearson r between each Haiku condition and human simplicity scores. Matches the rows `Haiku_full / simp+src / simp+ref / simp` (last row only across all 4 datasets) in the paper's Table 3.

In [ ]:
rows = []
for cond in HAIKU_CONDITIONS:
    row = {'Haiku condition': cond}
    for ds in HUMAN_DATASETS:
        row[ds] = corr(human[ds], cond, method='pearson', use_abs=True)
    rows.append(row)
table3 = pd.DataFrame(rows).set_index('Haiku condition')
table3.to_csv(OUTPUT_DIR / 'table3_haiku_conditions.csv', float_format='%.3f')
print('Table 3 - Haiku x conditions x dataset (|Pearson r|)')
display(table3.round(3))

## Table 4 - 5 LLM judges, direct scoring, output_only

Absolute Pearson r. Haiku uses our Stage-0 prompt convention; the other four models use the Kreutz et al. (2024) prompts. See `pairwise_position_bias.json` for the corresponding pairwise position-bias rates (Table 12).

In [ ]:
rows = []
for m in MODELS:
    row = {'Model': m.replace('Claude(simp)', 'Haiku 4.5')}
    for ds in HUMAN_DATASETS:
        row[ds] = corr(human[ds], m, method='pearson', use_abs=True)
    rows.append(row)
table4 = pd.DataFrame(rows).set_index('Model')
table4.to_csv(OUTPUT_DIR / 'table4_direct_5_models.csv', float_format='%.3f')
print('Table 4 - 5 LLM judges x 4 datasets, direct scoring, output_only (|Pearson r|)')
display(table4.round(3))

## Table 5 - ARTS + LR-ARTS Pearson r

Absolute Pearson r for each LLM judge on the three ARTS variants and their length-residualized counterparts. The paper's Table 5 shows the Haiku row plus baselines; the table below extends to all 5 models for completeness.

In [ ]:
rows = []
for m in MODELS:
    row = {'Model': m.replace('Claude(simp)', 'Haiku 4.5')}
    for label, store in [('ARTS94', arts['ARTS94']), ('LR-ARTS94', lr_arts['LR-ARTS94']),
                         ('ARTS300', arts['ARTS300']), ('LR-ARTS300', lr_arts['LR-ARTS300']),
                         ('ARTS3k', arts['ARTS3k']), ('LR-ARTS3k', lr_arts['LR-ARTS3k'])]:
        row[label] = corr(store, m, method='pearson', use_abs=True)
    rows.append(row)
table5 = pd.DataFrame(rows).set_index('Model')
table5.to_csv(OUTPUT_DIR / 'table5_arts_lr_arts.csv', float_format='%.3f')
print('Table 5 - 5 LLM judges x ARTS + LR-ARTS (|Pearson r|)')
display(table5.round(3))

## Table 10 - Full direct-scoring grid (Appendix B)

Pearson r for every (model, dataset, input-condition) cell we have. Non-Haiku models were only run on `output` (the paper's `-` indicates conditions not yet run for those models).

In [ ]:
CONDITION_LABELS = [
    ('output', None),                       # output_only
    ('+ source', 'Claude(simp+src)'),       # Haiku-only conditions live in named keys
    ('+ reference', 'Claude(simp+ref)'),
    ('+ source + ref', 'Claude(simp+src+ref)'),
]
ALL_DATASETS = [
    ('ARTS94', arts['ARTS94']),
    ('ARTS300', arts['ARTS300']),
    ('ARTS3k', arts['ARTS3k']),
    ('SDA', human['SDA']),
    ('ST-sent', human['ST-sent']),
    ('ST-para', human['ST-para']),
    ('D-Wiki', human['D-Wiki']),
]
DISPLAY_MODEL = {
    'Claude(simp)': 'Haiku 4.5',
    'GPT-4o-mini': 'GPT-4o-mini',
    'Gemma-4-E4B': 'Gemma-4-E4B',
    'Ministral-8B': 'Ministral-8B',
    'Llama-3.1-8B': 'Llama-3.1-8B',
}

rows = []
for ds_label, store in ALL_DATASETS:
    for cond_label, alt_key in CONDITION_LABELS:
        for model in MODELS:
            disp = DISPLAY_MODEL[model]
            if cond_label == 'output':
                key = model
            elif disp == 'Haiku 4.5':
                key = alt_key
            else:
                # non-Haiku models have no source / ref / src+ref columns
                key = None
            value = corr(store, key, method='pearson', use_abs=True) if key else float('nan')
            rows.append({'Dataset': ds_label, 'Condition': cond_label, 'Model': disp, 'Pearson r': value})
table10_long = pd.DataFrame(rows)
table10 = (table10_long
           .pivot_table(index=['Dataset', 'Condition'], columns='Model', values='Pearson r')
           [list(DISPLAY_MODEL.values())])  # column order
table10.to_csv(OUTPUT_DIR / 'table10_full_grid.csv', float_format='%.3f')
print('Table 10 - full direct-scoring grid (|Pearson r|)')
display(table10.round(3))

## Table 11 - LR-ARTS Spearman + Kendall (absolute)

Two correlation methods on the length-residualized ARTS variants. Haiku row plus the other 4 models.

In [ ]:
rows = []
for m in MODELS:
    disp = DISPLAY_MODEL[m]
    row = {'Model': disp}
    for label, store in lr_arts.items():
        row[(label, 'Spearman')] = corr(store, m, method='spearman', use_abs=True)
        row[(label, 'Kendall')] = corr(store, m, method='kendall', use_abs=True)
    rows.append(row)
table11 = pd.DataFrame(rows).set_index('Model')
table11.columns = pd.MultiIndex.from_tuples(table11.columns, names=['Dataset', 'Method'])
table11.to_csv(OUTPUT_DIR / 'table11_lr_arts_rank.csv', float_format='%.3f')
print('Table 11 - LR-ARTS rank correlations (|Spearman|, |Kendall|)')
display(table11.round(3))

## Table 12 - Pairwise prompting (Appendix C)

For each (model, dataset) cell on the 4 human-annotated datasets:

- **r**: signed Pearson r between the Bradley-Terry-mapped pairwise score and human scores.
- **delta-r**: signed change relative to the same model's direct (output_only) score.
- **bias**: fraction of pairs the model disagreed with itself on when the order was swapped (from the upstream manifest). Higher = more position-biased = less stable.

Ministral-8B on SDA shows '-' for r and delta-r because Bradley-Terry could not fit (the model returned empty on ~85% of pairwise prompts under the paper's prompt template). The bias rate is still reported.

In [ ]:
MODEL_TO_PAIRWISE = {
    'GPT-4o-mini':  'GPT-4o-mini (pairwise)',
    'Gemma-4-E4B':  'Gemma-4-E4B (pairwise)',
    'Ministral-8B': 'Ministral-8B (pairwise)',
    'Llama-3.1-8B': 'Llama-3.1-8B (pairwise)',
}

rows = []
for ds in HUMAN_DATASETS:
    store = human[ds]
    for direct_model, pairwise_key in MODEL_TO_PAIRWISE.items():
        # Signed r for direct + pairwise (within human-annotated, signs are already positive).
        r_direct = corr(store, direct_model, method='pearson', use_abs=False)
        r_pair = corr(store, pairwise_key, method='pearson', use_abs=False)
        delta_r = r_pair - r_direct if not (np.isnan(r_pair) or np.isnan(r_direct)) else float('nan')
        bias = position_bias.get(pairwise_key, {}).get(ds, float('nan'))
        rows.append({
            'Dataset': ds,
            'Model': direct_model,
            'Pairwise r': '-' if np.isnan(r_pair) else round(r_pair, 3),
            'delta-r': '-' if np.isnan(delta_r) else round(delta_r, 3),
            'Position bias': round(bias, 3) if not np.isnan(bias) else '-',
        })
table12 = pd.DataFrame(rows).set_index(['Dataset', 'Model'])
table12.to_csv(OUTPUT_DIR / 'table12_pairwise.csv')
print('Table 12 - pairwise prompting (signed r, delta-r vs direct, position bias)')
display(table12)

## Tables 13 and 14 - 95% confidence intervals

Pearson r with 95% CI for every LLM judge on every dataset. Reproduces the paper's Appendix E tables (extends with the 4 non-Haiku models). CI bounds are absolute values; the original signed bounds may differ for ARTS / LR-ARTS where the underlying human column is residualized complexity (see `src/data_loader.py` docstring).

In [ ]:
# Table 13 - ARTS
rows = []
for m in MODELS:
    disp = DISPLAY_MODEL[m]
    row = {'Model': disp}
    for ds, store in arts.items():
        row[ds] = corr_with_ci(store, m, method='pearson', use_abs=True)
    rows.append(row)
table13 = pd.DataFrame(rows).set_index('Model')
table13.to_csv(OUTPUT_DIR / 'table13_arts_ci.csv')
print('Table 13 - ARTS Pearson with 95% CI')
display(table13)

In [ ]:
# Table 14 - human-annotated datasets
rows = []
haiku_groups = list(HAIKU_CONDITIONS) + [m for m in MODELS if m != 'Claude(simp)']
for m in haiku_groups:
    disp = m if m.startswith('Claude') else m
    row = {'Metric': disp}
    for ds in HUMAN_DATASETS:
        row[ds] = corr_with_ci(human[ds], m, method='pearson', use_abs=True)
    rows.append(row)
# Pairwise rows too
for pw in PAIRWISE_MODELS:
    row = {'Metric': pw}
    for ds in HUMAN_DATASETS:
        row[ds] = corr_with_ci(human[ds], pw, method='pearson', use_abs=True)
    rows.append(row)
table14 = pd.DataFrame(rows).set_index('Metric')
table14.to_csv(OUTPUT_DIR / 'table14_human_ci.csv')
print('Table 14 - SDA / ST-sent / ST-para / D-Wiki Pearson with 95% CI')
display(table14)

## Summary

All tables have been written to `results/tables/`:

- `table3_haiku_conditions.csv`
- `table4_direct_5_models.csv`
- `table5_arts_lr_arts.csv`
- `table10_full_grid.csv`
- `table11_lr_arts_rank.csv`
- `table12_pairwise.csv`
- `table13_arts_ci.csv`
- `table14_human_ci.csv`

Each one mirrors a table in the paper. Spot-check against the PDF: e.g. SDA / Haiku in Table 4 should read 0.36.